# Solar Digital Twin — Live Monitor
**Order of operations:**
1. Run Cell 1 to install dependencies
2. Run Cell 2 to define everything
3. Open Wokwi in Chrome and press **Play**
4. Run Cell 3 to connect to HiveMQ and start receiving data
5. Run Cell 4 to see the live chart
6. Run Cell 5 when done to disconnect and save CSV

In [ ]:
%pip install paho-mqtt matplotlib numpy pandas --quiet

In [ ]:
import json
import threading
import numpy as np
import paho.mqtt.client as mqtt
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from collections import deque
from datetime import datetime
import pandas as pd

# ── MQTT config ───────────────────────────────────────────────────────────────
BROKER = "broker.hivemq.com"   # public broker — works with Wokwi free plan
PORT   = 1883
TOPIC  = "solar/twin_input"

# ── Ring buffers ──────────────────────────────────────────────────────────────
MAX_PTS  = 200
soc_real = deque(maxlen=MAX_PTS)
soc_twin = deque(maxlen=MAX_PTS)
volt_buf = deque(maxlen=MAX_PTS)
curr_buf = deque(maxlen=MAX_PTS)
log      = []

# ── Digital-twin model state ──────────────────────────────────────────────────
C_bat_Ah  = 2.0
V_bat     = 12.0
LOAD_W    = 2.5
soc_model = 100.0
_lock     = threading.Lock()

# ── MQTT callbacks ────────────────────────────────────────────────────────────
def on_connect(client, userdata, flags, rc):
    labels = {0:'Connected OK', 1:'Bad protocol', 2:'Bad client ID',
              3:'Server unavailable', 4:'Bad credentials', 5:'Not authorised'}
    print(f"MQTT: {labels.get(rc, f'rc={rc}')}")
    if rc == 0:
        client.subscribe(TOPIC)
        print(f"Subscribed to {TOPIC}")
        print("Waiting for data from Wokwi simulation...")

def on_message(client, userdata, msg):
    global soc_model
    try:
        data = json.loads(msg.payload.decode())
    except Exception as e:
        print(f"[Bad payload] {e}")
        return

    V        = float(data['V'])
    I        = float(data['I'])
    SOC      = float(data['SOC'])
    hw_alert = data.get('alert', '')

    P_net = (V * I) - LOAD_W
    delta = (P_net / V_bat) / C_bat_Ah / 3600 * 100

    with _lock:
        soc_model = float(np.clip(soc_model + delta, 0, 100))
        twin = soc_model

    sw_alert = ('CRITICAL' if twin < 15 else 'WARNING' if twin < 30 else 'OK')
    print(f"V={V:.2f}V  I={I*1000:.0f}mA  SOC_hw={SOC:.1f}%  SOC_twin={twin:.1f}%  [{sw_alert}]  hw={hw_alert}")

    soc_real.append(SOC)
    soc_twin.append(twin)
    volt_buf.append(V)
    curr_buf.append(I * 1000)
    log.append([datetime.now(), V, I, SOC, twin, sw_alert, hw_alert])

print('Definitions loaded. Run the next cell to connect.')

In [ ]:
# ── Connect to HiveMQ public broker ───────────────────────────────────────────
# Start the Wokwi simulation FIRST, then run this cell.
# Both Wokwi and Python connect to the same public broker.

client = mqtt.Client(client_id='PythonTwin_01', clean_session=True)
client.on_connect = on_connect
client.on_message = on_message

print(f"Connecting to {BROKER}:{PORT} ...")
client.connect(BROKER, PORT, keepalive=60)
client.loop_start()
print('Loop started — you should see Connected OK above within a few seconds.')

In [ ]:
# ── Live chart ────────────────────────────────────────────────────────────────
# If the window does not open, change '%matplotlib tk' to '%matplotlib qt'

%matplotlib tk

fig, (ax_soc, ax_vi) = plt.subplots(2, 1, figsize=(12, 7), sharex=False)
fig.suptitle('Solar Digital Twin — Live', fontsize=13, fontweight='bold')

ax_soc.set_title('State of Charge')
ax_soc.set_ylabel('SOC (%)')
ax_soc.set_ylim(0, 105)
ax_soc.set_xlim(0, MAX_PTS)
ax_soc.axhline(30, color='orange', linestyle='--', linewidth=0.8, label='Warning 30%')
ax_soc.axhline(15, color='red',    linestyle='--', linewidth=0.8, label='Critical 15%')
ax_soc.grid(True, alpha=0.3)
line_real, = ax_soc.plot([], [], color='steelblue', linewidth=1.5, label='Hardware SOC')
line_twin, = ax_soc.plot([], [], color='tomato',    linewidth=1.5, label='Twin SOC', linestyle='--')
ax_soc.legend(loc='upper right', fontsize=8)

ax_vi.set_title('Voltage & Current')
ax_vi.set_xlabel('Sample')
ax_vi.set_ylabel('Voltage (V)', color='steelblue')
ax_vi.set_ylim(10, 15)
ax_vi.set_xlim(0, MAX_PTS)
ax_vi.grid(True, alpha=0.3)
ax_vi.tick_params(axis='y', labelcolor='steelblue')
line_v, = ax_vi.plot([], [], color='steelblue', linewidth=1.2)

ax_i = ax_vi.twinx()
ax_i.set_ylabel('Current (mA)', color='seagreen')
ax_i.set_ylim(0, 300)
ax_i.tick_params(axis='y', labelcolor='seagreen')
line_i, = ax_i.plot([], [], color='seagreen', linewidth=1.2)

plt.tight_layout()

def update(_):
    x = list(range(len(soc_real)))
    if not x:
        return line_real, line_twin, line_v, line_i
    line_real.set_data(x, list(soc_real))
    line_twin.set_data(x, list(soc_twin))
    line_v.set_data(x, list(volt_buf))
    line_i.set_data(x, list(curr_buf))
    lo   = max(0, len(x) - MAX_PTS)
    span = max(MAX_PTS, len(x))
    for ax in [ax_soc, ax_vi, ax_i]:
        ax.set_xlim(lo, span)
    return line_real, line_twin, line_v, line_i

ani = animation.FuncAnimation(fig, update, interval=1000, blit=True, cache_frame_data=False)
plt.show()

In [ ]:
# ── Disconnect and save CSV ────────────────────────────────────────────────────
client.loop_stop()
client.disconnect()
print('MQTT disconnected.')

if log:
    df = pd.DataFrame(log, columns=['time','V','I_A','SOC_hw','SOC_twin','sw_alert','hw_alert'])
    df.to_csv('digital_twin_live.csv', index=False)
    print(f'Saved {len(df)} rows to digital_twin_live.csv')
    display(df.tail(10))
else:
    print('No data received — was the Wokwi simulation running?')